In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from tenacity import retry, stop_after_attempt, wait_exponential

from app.rag_simple.config import CHAT_MODEL
from app.rag_simple.retrieval.pipeline import retrieve

wait = wait_exponential(multiplier=1, min=10, max=240)


def _log_retry(retry_state):
    print(
        f"[retry] {retry_state.fn.__name__} attempt {retry_state.attempt_number} failed: "
        f"{retry_state.outcome.exception()!r} — retrying..."
    )

model = init_chat_model(CHAT_MODEL, temperature=0)

SYSTEM_PROMPT = """Ești un asistent cunoscător care răspunde la întrebări despre macrobiotică, \
sănătate naturistă și spiritualitate, pe baza unei baze de cunoștințe curatoriate (materiale despre macrobiotică).

Răspunsul tău va fi evaluat pentru acuratețe, relevanță și completitudine, deci răspunde doar la \
întrebare și răspunde-i complet. Dacă informația nu se regăsește în contextul de mai jos, spune \
clar că nu știi — nu inventa.

Context relevant din baza de cunoștințe:
{context}

Pe baza acestui context, răspunde la întrebarea userului. Fii precis, relevant și complet."""


def format_context(chunks: list[Document]) -> str:
    return "\n\n".join(
        f"Extras din {chunk.metadata.get('source', '?')}:\n{chunk.page_content}" for chunk in chunks
    )


def make_messages(question: str, history: list[dict], chunks: list[Document]) -> list[dict]:
    system_prompt = SYSTEM_PROMPT.format(context=format_context(chunks))
    return (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )


@retry(wait=wait, stop=stop_after_attempt(5), before_sleep=_log_retry, reraise=True)
def answer_question(question: str, history: list[dict] | None = None) -> tuple[str, list[Document]]:
    """Answer a question using RAG. Returns (answer, retrieved chunks)."""
    history = history or []
    chunks = retrieve(question, history)
    messages = make_messages(question, history, chunks)
    response = model.invoke(messages)
    return response.content, chunks


In [5]:
question = "Ce este macrobiotica?"
answer_question(question, history=[])

('Macrobiotica este un mod de viață tradițional, bazat pe bunul simț și pe aplicarea principiilor filosofiei orientale vechi de peste 5.000 de ani. Ea reprezintă arta dobândirii unei adevărate longevități și a unei vieți fericite prin sănătate, punând în practică o concepție dialectică asupra universului, care echilibrează elementele Yin și Yang în alimentație și stil de viață.\n\nMacrobiotica nu este o medicină empirică, mistică sau paliativă, ci o disciplină care integrează armonia între corp, spirit și suflet, condiție indispensabilă pentru o viață fericită. Ea provine în special din tradițiile asiatice, în special japoneze, dar reflectă înțelepciunea vechilor culturi din întreaga lume.\n\nPractic, macrobiotica implică o alimentație echilibrată, bazată pe cereale integrale, legume și alte alimente naturale, preparate cu atenție pentru a menține echilibrul Yin-Yang. Mestecatul corect, recunoștința pentru hrană și o atitudine spirituală pozitivă sunt părți esențiale ale acestei practi